# Cross Validation

A base de dados estudada neste projeto contém informações sobre variáveis ambientais coletadas para a detecção de incêndios. O objetivo é utilizar técnicas de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação na previsão da ocorrência de um incêndio com base nas variáveis fornecidas.

A base de dados contém as seguintes variáveis:
- Unnamed:0: Índice (não é uma variável útil para o modelo)
- UTC: Tempo em Segundos UTC
- Temperature[C]: Temperatura do Ar (em graus Celsius)
- Humidity[%]: Umidade do Ar (em porcentagem)
- TVOC[ppb]: Total de Compostos Orgânicos Voláteis (medido em partes por bilhão)
- eCO2[ppm]: Concentração equivalente de CO2 (medido em partes por milhão)
- Raw H2: Hidrogênio molecular bruto, não compensado
- Raw Ethanol: Etanol gasoso bruto
- Pressure[hPA]: Pressão do Ar (em hectopascais)
- PM1.0: Material particulado de tamanho < 1,0 µm
- PM2.5: Material particulado de tamanho >1,0 µm e < 2,5 µm
- NC0.5: Concentração numérica de material particulado de tamanho < 0,5 µm
- NC1.0: Concentração numérica de material particulado de tamanho 0,5 µm < 1,0 µm
- NC2.5: Concentração numérica de material particulado de tamanho 1,0 µm < 2,5 µm
- CNT: Contador de amostras

E a variável alvo:
- Fire Alarm: Indicador binário de incêndio (1 se houver incêndio, 0 caso contrário)

### **Configurações**

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('smoke_detection_iot.csv')


### **Exploração inicial dos dados** 

A exploração mostra que os dados foram corretamente importados, que não há valores nulos nem faltantes. As de mais características do banco já foram tratadas previamente. O Nome da coluna 'Fire Alarm', por ter um espaço no nome, pode causar problemas. Por isso, o nome foi alterado para 'Fire_Alarm'.

In [ ]:
print(df.head(2))
print(df.columns)

   Unnamed: 0         UTC  Temperature[C]  Humidity[%]  TVOC[ppb]  eCO2[ppm]  \
0           0  1654733331          20.000        57.36          0        400   
1           1  1654733332          20.015        56.67          0        400   

   Raw H2  Raw Ethanol  Pressure[hPa]  PM1.0  PM2.5  NC0.5  NC1.0  NC2.5  CNT  \
0   12306        18520        939.735    0.0    0.0    0.0    0.0    0.0    0   
1   12345        18651        939.744    0.0    0.0    0.0    0.0    0.0    1   

   Fire Alarm  
0           0  
1           0  
Index(['Unnamed: 0', 'UTC', 'Temperature[C]', 'Humidity[%]', 'TVOC[ppb]',
       'eCO2[ppm]', 'Raw H2', 'Raw Ethanol', 'Pressure[hPa]', 'PM1.0', 'PM2.5',
       'NC0.5', 'NC1.0', 'NC2.5', 'CNT', 'Fire Alarm'],
      dtype='object')


In [3]:
df.rename(columns={'Fire Alarm': 'Fire_Alarm'}, inplace=True)
print(df.columns)

Index(['Unnamed: 0', 'UTC', 'Temperature[C]', 'Humidity[%]', 'TVOC[ppb]',
       'eCO2[ppm]', 'Raw H2', 'Raw Ethanol', 'Pressure[hPa]', 'PM1.0', 'PM2.5',
       'NC0.5', 'NC1.0', 'NC2.5', 'CNT', 'Fire_Alarm'],
      dtype='object')


In [4]:
df.isnull().sum()

Unnamed: 0        0
UTC               0
Temperature[C]    0
Humidity[%]       0
TVOC[ppb]         0
eCO2[ppm]         0
Raw H2            0
Raw Ethanol       0
Pressure[hPa]     0
PM1.0             0
PM2.5             0
NC0.5             0
NC1.0             0
NC2.5             0
CNT               0
Fire_Alarm        0
dtype: int64

In [5]:
df.dtypes

Unnamed: 0          int64
UTC                 int64
Temperature[C]    float64
Humidity[%]       float64
TVOC[ppb]           int64
eCO2[ppm]           int64
Raw H2              int64
Raw Ethanol         int64
Pressure[hPa]     float64
PM1.0             float64
PM2.5             float64
NC0.5             float64
NC1.0             float64
NC2.5             float64
CNT                 int64
Fire_Alarm          int64
dtype: object

### **Exploração de dados para a escolha do modelo**

Para escolher o melhor modelo de aprendizagem de máquina para esses dados, a principal característica considerada foi de que a variável alvo é binária (0/1). Isso elimina os modelos de regressão linear e polinomial, já que não há como modelar uma curva contínua que descreva o comportamento com uma variável discontínua. Por motivos similares, foram descartados também os modelos de agrupamento, como o Kmeans.

Após explorar os dados, optou-se por testar os modelos de Árvore de Decisão, Random Florest e Naive-Bayes. A escolha foi embasada na hipótese de que mais de uma variável são responsáveis pelos incendios. Ao analisar a correlção etre as variáveis e o alarme de incêndio, listadas abaixo, destaca-se a correlação com o CNT. As demais tem uma correlção menor, mas que se relacionam em o alarme em intervalos similares. Isso indica que, apesar de um peso menor, elas tem influência na variável alvo.

In [68]:
corr = df.corr()
print(corr['Fire_Alarm'].sort_values(ascending=False))

Fire_Alarm        1.000000
CNT               0.673762
Humidity[%]       0.399846
Pressure[hPa]     0.249797
Raw H2            0.107007
NC2.5            -0.057707
NC1.0            -0.082828
PM2.5            -0.084916
eCO2[ppm]        -0.097006
PM1.0            -0.110552
NC0.5            -0.128118
Temperature[C]   -0.163902
TVOC[ppb]        -0.214743
Raw Ethanol      -0.340652
Unnamed: 0       -0.361351
UTC              -0.389404
Name: Fire_Alarm, dtype: float64


### Cross-variation

In [30]:
from sklearn.tree import DecisionTreeClassifier 

### Árvore de decisão

In [44]:
y = df['Fire_Alarm']
x = df.drop('Fire_Alarm', axis=1)
print(x.shape, y.shape)


(62630, 15) (62630,)


In [98]:
modelo_arvore = DecisionTreeClassifier(criterion = 'gini', random_state = 0)
folds = 10
crossvalidation = KFold(n_splits=folds, shuffle=True, random_state=folds)
modelo_arvore_final = cross_val_score(modelo_arvore, x, y, cv = folds)
# Avaliação
pontuacoes_arvore = cross_val_score(modelo_arvore, x, y, cv=crossvalidation)
pontuacoes_arvore_media = pontuacoes_arvore.mean()

### Random forest

In [80]:
from sklearn.ensemble import RandomForestClassifier
modelo_rf = RandomForestClassifier(random_state=42)

In [101]:
folds = 5
crossvalidation = KFold(n_splits=folds, shuffle=True, random_state=5)
modelo_rf_final = cross_val_score(modelo_rf, x, y, cv = folds)
# Avaliação
pontuacoes_rf = cross_val_score(modelo_rf, x, y, cv=crossvalidation)
pontuacoes_rf_media = pontuacoes_rf.mean()

KeyboardInterrupt: 

### Naive-Bayes

In [85]:
from sklearn.naive_bayes import GaussianNB
modelo_naive = GaussianNB()

In [99]:
folds = 10
crossvalidation = KFold(n_splits=folds, shuffle=True, random_state=folds)
modelo_naive_final = cross_val_score(modelo_naive, x, y, cv = folds)
#Avaliação
pontuacoes_naive = cross_val_score(modelo_naive, x, y, cv=crossvalidation)
pontuacoes_naive_media = pontuacoes_naive.mean()

### Comparação

A pontuação média de desempenho de cada modelo segue abaixo:

In [102]:
print(f'Pontuação Média de cada modelo:\nAD: {pontuacoes_arvore_media: .5f}\nRF: {pontuacoes_rf_media: .5f}\nNB: {pontuacoes_naive_media: .5f}')

Pontuação Média de cada modelo:
AD:  0.99984
RF:  0.99994
NB:  0.83632


O melhor desempenho foi o do modelo Random Forest, mesmo com um número menor de folds. A diferença entre os resultados para árvore de decisão e random forest é pequeno, indicando que não há muitas diferenças entre s previsões dos dois. A diferença é maior para o modelo Naive-Bayes, por isso o modelo será descartado.